## 라이브러리 호출 및 데이터 로드

In [40]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json
from pprint import pprint

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

In [41]:
# 데이터 로드
df_meta = pd.read_csv('./유저단위_게임데이터_상위랭커보존-데이터복구완료.csv')

In [42]:
df_meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 396239 entries, 0 to 396238
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gameid               396239 non-null  str    
 1   gameduration         396239 non-null  float64
 2   level                396239 non-null  int64  
 3   lastround            396239 non-null  int64  
 4   ranked               396239 non-null  int64  
 5   ingameduration       396239 non-null  float64
 6   combination          396239 non-null  str    
 7   champion             396239 non-null  str    
 8   user_tier            396239 non-null  str    
 9   season               396239 non-null  str    
 10  user_id              396239 non-null  str    
 11  flag_1               396239 non-null  int64  
 12  flag_2               396239 non-null  int64  
 13  mix_flag             396239 non-null  str    
 14  combination_rebuild  396239 non-null  str    
dtypes: float64(2), int64(5), str

---
## 1. 시너지명 정리
- 시너지명에 'Set3_'가 포함된 경우, 삭제

In [43]:
# combination_rebuild 컬럼의 데이터 타입을 딕셔너리로 변경
df_meta['combination_rebuild'] = df_meta['combination_rebuild'].apply(ast.literal_eval)

# 데이터형이 변경되었는지 확인
type(df_meta['combination_rebuild'].iloc[0]) # dict

dict

In [44]:
# 전체 키 목록 확인하는 함수 정의
# set(): 중복을 허용하지 않는 자료형(집합)
# .update(): set에 여러 값을 한 번에 추가하는 메서드
def get_all_keys(df, col):
    all_keys = set()
    for value in df[col]:
        all_keys.update(value.keys())
    return all_keys

# 함수 실행 (접두사 'Set3_' 제거하기 전)
print(get_all_keys(df_meta, 'combination_rebuild'))

{'Cybernetic', 'Vanguard', 'Set3_Mystic', 'Protector', 'Set3_Blademaster', 'Rebel', 'Chrono', 'Valkyrie', 'Demolitionist', 'Infiltrator', 'Set3_Sorcerer', 'DarkStar', 'Set3_Celestial', 'ManaReaver', 'StarGuardian', 'Set3_Brawler', 'MechPilot', 'SpacePirate', 'Starship', 'Sniper', 'Blaster', 'Set3_Void', 'Mercenary'}


In [45]:
# 접두사인 'Set3_'를 삭제하는 함수 정의
def remove_set3_prefix(x):
    result = {}
    for key, value in x.items():
        if 'Set3_' in key:
            new_key = key.replace('Set3_', '')
        else:
            new_key = key
        result[new_key] = value
    return result

In [46]:
# 정의한 함수를 사용해 시너지명 정리
# apply(): 행마다 함수를 적용함
df_meta['combination_rebuild'] = df_meta['combination_rebuild'].apply(remove_set3_prefix)

In [47]:
# 접두사 'Set3_' 제거한 결과 확인
print(get_all_keys(df_meta, 'combination_rebuild'))

{'Brawler', 'Cybernetic', 'Vanguard', 'Sorcerer', 'Protector', 'Rebel', 'Chrono', 'Valkyrie', 'Demolitionist', 'Infiltrator', 'Celestial', 'DarkStar', 'Mystic', 'Void', 'ManaReaver', 'StarGuardian', 'MechPilot', 'SpacePirate', 'Blademaster', 'Starship', 'Sniper', 'Blaster', 'Mercenary'}


---
## 2. 비활성화된 시너지 삭제
- 게임이 진행되는 동안, 활성화되지 않은 시너지 정보 삭제

In [48]:
# 시너지별 활성화 기준(Threshold) 정의
synergy_thresholds = {
    # 직업 시너지
    'Blademaster': [3, 6, 9],
    'ManaReaver': [2],
    'Sorcerer': [2, 4, 6, 8],
    'Vanguard': [2, 4],
    'Protector': [2, 4, 6],
    'Mystic': [2, 4],
    'Brawler': [2, 4],
    'Mercenary': [1],
    'Starship': [1],
    'Infiltrator': [2, 4, 6],
    'Sniper': [2],
    'Blaster': [2, 4],
    'Demolitionist': [2],
    
    # 계열 시너지
    'Void': [3],
    'MechPilot': [3],
    'Rebel': [3, 6, 9],
    'Valkyrie': [2],
    'StarGuardian': [3, 6],
    'Cybernetic': [3, 6],
    'Chrono': [2, 4, 6],
    'DarkStar': [3, 6, 9],
    'SpacePirate': [2, 4],
    'Celestial': [2, 4, 6]
}

In [49]:
# 활성화 레벨 계산 함수 정의
def get_active_level(count, thresholds):
    active = 0
    for t in thresholds:
        if count >= t:
            active = t
        else:
            break
    return active

# 딕셔너리(dict) 형태로 활성화된 시너지만 반환하는 함수 정의
def parse_active_synergies_to_dict(comb_dict):
    # 데이터가 이미 딕셔너리 형태이므로 ast.literal_eval 불필요. 빈 딕셔너리인지 확인
    if not isinstance(comb_dict, dict) or not comb_dict:
        return {}
        
    active_res = {}
    for syn, count in comb_dict.items():
        if syn in synergy_thresholds:
            active_lvl = get_active_level(count, synergy_thresholds[syn])
            # 활성화 레벨이 0보다 큰 경우(최소 조건 달성)에만 딕셔너리에 추가
            if active_lvl > 0:
                active_res[syn] = active_lvl
                
    # 결과도 깔끔하게 딕셔너리 형태로 반환 (json 변환 불필요)
    return active_res

In [50]:
# 'active_synergies' 컬럼을 추가하여 활성화된 시너지 딕셔너리만 따로 확인
df_meta['active_synergies'] = df_meta['combination_rebuild'].apply(parse_active_synergies_to_dict)

In [51]:
# 결과 확인
display(df_meta[['combination_rebuild', 'active_synergies']].head())

,combination_rebuild,active_synergies
0,"{'Cybernetic': 1, 'Demolitionist': 1, 'Infiltrator': 1, 'Rebel': 1, 'Brawler': 1, 'Celestial': 1, 'Void': 1, 'Sniper': 1}",{}
1,"{'Blaster': 1, 'Chrono': 1, 'Cybernetic': 4, 'Demolitionist': 1, 'Rebel': 1, 'Blademaster': 2, 'Brawler': 1, 'Sorcerer': 1, 'Void': 1, 'Valkyrie': 1, 'Vanguard': 2}","{'Cybernetic': 3, 'Vanguard': 2}"
2,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 2, 'Infiltrator': 1, 'Mercenary': 1, 'Blademaster': 1, 'Mystic': 1, 'Valkyrie': 1}",{'Mercenary': 1}
3,"{'DarkStar': 1, 'Protector': 2, 'Blademaster': 1, 'Celestial': 5, 'Mystic': 1, 'Sniper': 1, 'StarGuardian': 2, 'Vanguard': 2}","{'Protector': 2, 'Celestial': 4, 'Vanguard': 2}"
4,"{'Blaster': 1, 'Chrono': 5, 'DarkStar': 3, 'Protector': 1, 'Blademaster': 1, 'Brawler': 1, 'Sorcerer': 2, 'Sniper': 2}","{'Chrono': 4, 'DarkStar': 3, 'Sorcerer': 2, 'Sniper': 2}"


### 2-1. ranked 관련된 flag 칼럼 2개 생성
- `top4_flag` = ranked 칼럼 값이 1~4 -> True / 5~8 > False
- `ranked_1` = ranked 칼럼 값이 1 -> True / 2~8 > False

In [52]:
df_meta['top4_flag'] = df_meta['ranked'].isin(range(1, 5))
df_meta['ranked_1'] = (df_meta['ranked'] == 1)

In [53]:
# ranked 값별로 칼럼 값 확인
print(df_meta.groupby('ranked')[['top4_flag', 'ranked_1']].first())

        top4_flag  ranked_1
ranked                     
1            True      True
2            True     False
3            True     False
4            True     False
5           False     False
6           False     False
7           False     False
8           False     False


### 2-2. 2차 전처리 완료 파일 csv 추출

In [54]:
# combination 데이터를 복구한 뒤, csv 파일로 저장
df_meta.to_csv("./유저단위_게임데이터_상위랭커보존-2차 전처리 완료(match).csv", index=False, encoding='utf-8-sig')

---
## 3. champion 테이블 전처리

### 3-1. champion 테이블 내 class 컬럼에 존재하는 오타 수정
- `Infiltrato` → `Infiltrator` 오타 수정

In [55]:
# champion 데이터 로드
champions = pd.read_csv('./tft_game_dataset/TFT_Champion_CurrentVersion.csv')

In [56]:
# Infiltrator 관련 전체 조회 (오타 포함)
infiltrator_all = champions[
    champions['class'].str.contains('Infiltrat', case=False, na=False)
][['name', 'class']]

print(f"Infiltrat* 포함 Champion 개수: {len(infiltrator_all)}")
print(infiltrator_all)

Infiltrat* 포함 Champion 개수: 5
      name            class
18   shaco  ['Infiltrator']
30    ekko   ['Infiltrato']
45   kaisa  ['Infiltrator']
46  khazix  ['Infiltrator']
51    fizz  ['Infiltrator']


In [57]:
# 정상 값만 출력(Infiltrator)
infiltrator_correct = champions[
    champions['class'].str.contains(r'\bInfiltrator\b', case=False, na=False)
][['name', 'class']]
print(f"\n정상 Infiltrator 개수: {len(infiltrator_correct)}")


정상 Infiltrator 개수: 4


In [58]:
# 오타만 따로 추출 (정상값 제외)
infiltrator_typo = infiltrator_all[
    ~infiltrator_all['class'].str.contains(r'\bInfiltrator\b', case=False, na=False)
]

print(f"\n오타 의심 항목")
print(infiltrator_typo)


오타 의심 항목
    name           class
30  ekko  ['Infiltrato']


In [59]:
# 오타 수정
champions['class'] = champions['class'].str.replace(
    r'\bInfiltrato\b', 'Infiltrator', regex=True
)

In [60]:
# 수정 후 확인
infiltrator_after = champions[champions['class'].str.contains('Infiltrator', na=False)][['name', 'class']]
print(f"Infiltrator 포함 Champion 개수: {len(infiltrator_after)}")
print(infiltrator_after)

Infiltrator 포함 Champion 개수: 5
      name            class
18   shaco  ['Infiltrator']
30    ekko  ['Infiltrator']
45   kaisa  ['Infiltrator']
46  khazix  ['Infiltrator']
51    fizz  ['Infiltrator']


### 3-2. 컬럼명 및 해당값 소문자로 변환

In [61]:
# 컬럼명 소문자 통일
champions.columns = champions.columns.str.lower()

champions.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             52 non-null     str    
 1   cost             52 non-null     int64  
 2   health           52 non-null     int64  
 3   defense          52 non-null     int64  
 4   attack           52 non-null     int64  
 5   attack_range     52 non-null     int64  
 6   speed_of_attack  52 non-null     float64
 7   dps              52 non-null     int64  
 8   skill_name       52 non-null     str    
 9   skill_cost       52 non-null     str    
 10  origin           52 non-null     str    
 11  class            52 non-null     str    
dtypes: float64(1), int64(6), str(5)
memory usage: 5.0 KB


### 3-3. 새로운 데이프레임 제작
- match 데이터에서 챔피언과 관련된 정보만 담은 데이터 프레임 제작 (`df_meta`)

In [62]:
# champion 컬럼의 데이터 타입을 딕셔너리로 변경
df_meta['champion'] = df_meta['champion'].apply(ast.literal_eval)

In [63]:
type(df_meta['champion'].iloc[0])

dict

In [74]:
# match 데이터에서 챔피언과 관련된 정보만 담은 데이터 프레임 제작
champion_data = []

# 매치 데이터에서 가져올 컬럼 목록
for idx, row in df_meta.iterrows():
    # row는 현재 행의 데이터를 담고 있는 Series
    game_id = row['gameid']
    user_tier = row['user_tier']
    ranked = row['ranked']
    user_id = row['user_id']
    flag_1 = row['flag_1']
    flag_2 = row['flag_2']
    top4_flag = row['top4_flag']
    ranked_1 = row['ranked_1']
    
    # 각 챔피언별 행 생성
    for champion_name, champion_info in row['champion'].items():
        items = champion_info.get('items', [])  # 아이템 리스트
        star = champion_info.get('star', 0)     # 챔피언 레벨
        
        champion_data.append({
            'gameid': game_id,
            'user_id': user_id,
            'user_tier': user_tier,
            'ranked': ranked,
            'ranked_1': ranked_1,
            'top4_flag': top4_flag,
            'champion_name': champion_name,
            'items': items,    
            'star': star,
            'flag_1': flag_1,
            'flag_2': flag_2
        })

In [75]:
# 데이터프레임 생성
champion_table = pd.DataFrame(champion_data)

In [76]:
# 결과 확인
champion_table.head()

,gameid,user_id,user_tier,ranked,ranked_1,top4_flag,champion_name,items,star,flag_1,flag_2
0,KR_4291707834,KR-USER-1,platinum,5,False,False,Ziggs,[7],1,0,0
1,KR_4291707834,KR-USER-1,platinum,5,False,False,Ashe,[9],1,0,0
2,KR_4291707834,KR-USER-1,platinum,5,False,False,ChoGath,[6],1,0,0
3,KR_4291707834,KR-USER-1,platinum,5,False,False,Ekko,[1],1,0,0
4,KR_4291707834,KR-USER-2,platinum,3,False,True,Ziggs,[24],3,0,0


In [77]:
# Champion 테이블에 Champions 파일의 정보 추가
# 챔피언의 이름을 소문자로 통일 
champion_table['champion_name'] = champion_table['champion_name'].str.strip().str.lower()
champions['name'] = champions['name'].str.strip().str.lower()

# champion_table과 champions 테이블 merge
champion_with_details = champion_table.merge(
    champions[['name', 'cost', 'origin', 'class']],
    left_on='champion_name',  
    right_on='name',
    how='left'  
)

In [78]:
# 결과 확인
champion_with_details.head()

,gameid,user_id,user_tier,ranked,ranked_1,top4_flag,champion_name,items,star,flag_1,flag_2,name,cost,origin,class
0,KR_4291707834,KR-USER-1,platinum,5,False,False,ziggs,[7],1,0,0,ziggs,1,Rebel,['Demolitionist']
1,KR_4291707834,KR-USER-1,platinum,5,False,False,ashe,[9],1,0,0,ashe,3,Celestial,['Sniper']
2,KR_4291707834,KR-USER-1,platinum,5,False,False,chogath,[6],1,0,0,chogath,4,Void,['Brawler']
3,KR_4291707834,KR-USER-1,platinum,5,False,False,ekko,[1],1,0,0,ekko,5,Cybernetic,['Infiltrator']
4,KR_4291707834,KR-USER-2,platinum,3,False,True,ziggs,[24],3,0,0,ziggs,1,Rebel,['Demolitionist']


In [80]:
champion_with_details.dtypes

# type(champion_with_details['items'].iloc[0]) # list
# type(champion_with_details['class'].iloc[0]) # str

gameid              str
user_id             str
user_tier           str
ranked            int64
ranked_1           bool
top4_flag          bool
champion_name       str
items            object
star              int64
flag_1            int64
flag_2            int64
name                str
cost              int64
origin              str
class               str
dtype: object

In [81]:
# Class 칼럼 값 표기 정제
# 리스트형태의 문자열 -> 문자열
champion_with_details['class'] = champion_with_details['class'].apply(
    lambda x: ', '.join(ast.literal_eval(x)) if isinstance(x, str) else ', '.join(x if isinstance(x, list) else [])
)

###  3-4. champion explode table (1차) 파일 csv 추출

In [82]:
champion_with_details.to_csv("./유저단위_게임데이터_상위랭커보존-explode_champion_1.csv", index=False, encoding='utf-8-sig')

### 3-5. combination 정보가 담긴 새로운 데이터프레임 제작

In [83]:
# Combination 테이블 생성
combination_data = []

# 매치 데이터에서 가져올 컬럼 목록
for idx, row in df_meta.iterrows():
    # row는 현재 행의 데이터를 담고 있는 Series
    game_id = row['gameid']
    user_tier = row['user_tier']
    ranked = row['ranked']
    user_id = row['user_id']
    flag_1 = row['flag_1']
    flag_2 = row['flag_2']
    active_synergies = row['active_synergies']
    top4_flag = row['top4_flag']
    ranked_1 = row['ranked_1']

    for synergy, synergy_val in row['combination_rebuild'].items():

        combination_data.append({
            'gameid': game_id,
            'user_id': user_id,
            'user_tier': user_tier,
            'ranked': ranked,
            'top4_flag': top4_flag,
            'ranked_1': ranked_1,
            'flag_1': flag_1,
            'flag_2': flag_2,
            'active_synergies': active_synergies,
            'synergy': synergy,
            'synergy_val': synergy_val
        })
combination_table = pd.DataFrame(combination_data)

In [84]:
# 결과 확인
display(combination_table)
print('='*50)
print(type(combination_table['active_synergies'].iloc[0])) # dict

,gameid,user_id,user_tier,ranked,top4_flag,ranked_1,flag_1,flag_2,active_synergies,synergy,synergy_val
0,KR_4291707834,KR-USER-1,platinum,5,False,False,0,0,{},Cybernetic,1
1,KR_4291707834,KR-USER-1,platinum,5,False,False,0,0,{},Demolitionist,1
2,KR_4291707834,KR-USER-1,platinum,5,False,False,0,0,{},Infiltrator,1
3,KR_4291707834,KR-USER-1,platinum,5,False,False,0,0,{},Rebel,1
4,KR_4291707834,KR-USER-1,platinum,5,False,False,0,0,{},Brawler,1
...,...,...,...,...,...,...,...,...,...,...,...
3361659,KR_4357265434,KR-USER-399998,challenger,8,False,False,0,0,"{'ManaReaver': 2, 'Protector': 2, 'Celestial': 4, 'SpacePirate': 2}",Protector,2
3361660,KR_4357265434,KR-USER-399998,challenger,8,False,False,0,0,"{'ManaReaver': 2, 'Protector': 2, 'Celestial': 4, 'SpacePirate': 2}",Celestial,4
3361661,KR_4357265434,KR-USER-399998,challenger,8,False,False,0,0,"{'ManaReaver': 2, 'Protector': 2, 'Celestial': 4, 'SpacePirate': 2}",Sniper,1
3361662,KR_4357265434,KR-USER-399998,challenger,8,False,False,0,0,"{'ManaReaver': 2, 'Protector': 2, 'Celestial': 4, 'SpacePirate': 2}",SpacePirate,2


<class 'dict'>


### 3-6.combination 정보가 담긴 새로운 데이터프레임 제작

In [85]:
combination_table.to_csv("./유저단위_게임데이터_상위랭커보존-explode_combination_1.csv", index=False, encoding='utf-8-sig')

---
## 4. items 테이블 전처리

### 4-1. items 정보를 병합한 데이터프레임 제작

In [86]:
# 아이템 데이터 로드
items_table = pd.read_csv("./tft_game_dataset/TFT_Item_CurrentVersion.csv")

In [87]:
# 아이템 이름 매핑을 위해 딕셔너리 생성
item_dict = dict(zip(items_table['id'], items_table['name']))

#Champion 테이블에 Item 정보 추가
champion_table_with_items_name = champion_with_details.copy()

# 아이템 개수 계산
champion_table_with_items_name['item_count'] = champion_table_with_items_name['items'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# explode
champion_table_item_exploded = champion_table_with_items_name.explode('items', ignore_index=True)

# 숫자 변환 
champion_table_item_exploded['item_id'] = pd.to_numeric(
    champion_table_item_exploded['items'], errors='coerce'
).astype('Int64')

# 매핑
champion_table_item_exploded['item_name'] = champion_table_item_exploded['item_id'].map(item_dict)

# # 정리
# champion_table_exploded = champion_table_exploded[[
#     'gameId', 'tier', 'Ranked', 'champion_name', 'item_id', 'item_name', 'star', 'item_count'
# ]]

In [88]:
champion_table_item_exploded.shape

(4862002, 18)

In [89]:
# csv 파일로 저장
champion_table_item_exploded.to_csv("./유저단위_게임데이터_상위랭커보존-explode_items_1.csv", index=False, encoding='utf-8-sig')

- 파일경로 가장 앞 부분은 `../../`로 변경하시면 csv가 저장된 위치에서 내부 개인작업 폴더로 로드 가능하십니다.
- 1차 파생테이블에서 파생컬럼을 새롭게 만드시는 경우, 의미도 함께 마크다운 or 주석으로 남겨주세요..
(파일 통합할 때..그것이 없으면 아무리 회의때 기록을 하더라도 결국 여러분을 주피터 라이브쉐어로 초대해서 정리해야 하는..리소스 낭비가 생길 수 있습니다ㅠㅠㅠ)